## Backtest Multi-Nível

**Correção aplicada:** `Media_3` agora usa `.shift(1).rolling(3).mean()` — sem data leakage.

#### Regra prática (mercado)
| Erro %   | Interpretação  |
|----------|----------------|
| < 3%     | Excelente 🔥   |
| 3% – 5%  | Muito bom      |
| 5% – 10% | Ok             |
| > 10%    | Ruim           |


In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from datetime import datetime

engine = create_engine("sqlite:///../data/DBVendas.db")


In [2]:
print("📊 Carregando dados...")

query = """
SELECT
    pd.Data_Venda,
    iv.ID_Produto,
    iv.Qtde,
    p.ID_Subcategoria,
    s.ID_Categoria,
    pd.ID_Canal
FROM itens_vendas iv
JOIN vendas pd ON iv.ID_Pedido = pd.ID_Pedido
JOIN produtos p ON iv.ID_Produto = p.ID_Produto
JOIN subcategorias s ON p.ID_Subcategoria = s.ID_Subcategoria
"""

df = pd.read_sql(query, engine)
df['Data_Venda'] = pd.to_datetime(df['Data_Venda'])

df['Ano'] = df['Data_Venda'].dt.year
df['Mes'] = df['Data_Venda'].dt.month

print(f"✅ Dados: {df.shape}")


📊 Carregando dados...
✅ Dados: (100000, 8)


In [3]:
# ==============================
# FUNÇÃO DE PREPARO (SEM LEAKAGE)
# ==============================

def preparar_dados(df, colunas_group):

    df_agg = df.groupby(colunas_group + ['Ano', 'Mes'])['Qtde'].sum().reset_index()

    df_agg['Data'] = pd.to_datetime(
        df_agg['Ano'].astype(str) + '-' + df_agg['Mes'].astype(str) + '-01'
    )

    df_agg = df_agg.sort_values(colunas_group + ['Data'])

    # Lags (apenas valores passados — sem leakage)
    df_agg['Lag_1'] = df_agg.groupby(colunas_group)['Qtde'].transform(lambda x: x.shift(1))
    df_agg['Lag_2'] = df_agg.groupby(colunas_group)['Qtde'].transform(lambda x: x.shift(2))
    df_agg['Lag_3'] = df_agg.groupby(colunas_group)['Qtde'].transform(lambda x: x.shift(3))

    # ✅ CORRIGIDO: shift(1) ANTES do rolling para não incluir o valor atual
    df_agg['Media_3'] = (
        df_agg.groupby(colunas_group)['Qtde']
        .transform(lambda x: x.shift(1).rolling(3).mean())
    )

    # Sazonalidade cíclica
    df_agg['Mes_sin'] = np.sin(2 * np.pi * df_agg['Mes'] / 12)
    df_agg['Mes_cos'] = np.cos(2 * np.pi * df_agg['Mes'] / 12)

    df_agg = df_agg.dropna()

    return df_agg


In [4]:
# ==============================
# FUNÇÃO BACKTEST (walk-forward)
# ==============================

def rodar_backtest(df_agg, colunas_group, nivel_nome):

    print(f"\n🚀 Backtest nível: {nivel_nome}")

    resultados = []
    anos = sorted(df_agg['Ano'].unique())

    for ano_teste in anos[3:]:

        train = df_agg[df_agg['Ano'] < ano_teste]
        test  = df_agg[df_agg['Ano'] == ano_teste]

        if len(test) == 0:
            continue

        X_train = train.drop(columns=['Qtde', 'Data'])
        y_train = train['Qtde']
        X_test  = test.drop(columns=['Qtde', 'Data'])
        y_test  = test['Qtde']

        model = HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.05, max_depth=6, random_state=42
        )
        model.fit(X_train, y_train)

        y_pred   = model.predict(X_test)
        mae      = mean_absolute_error(y_test, y_pred)
        erro_pct = (mae / y_test.mean()) * 100

        print(f"📅 {ano_teste} | MAE: {mae:.2f} | Erro %: {erro_pct:.2f}")

        resultados.append({
            'Nivel'  : nivel_nome,
            'Ano'    : ano_teste,
            'MAE'    : mae,
            'Erro_%' : erro_pct,
            'Qtd'    : len(test)
        })

    return pd.DataFrame(resultados)


In [5]:
# ==============================
# EXECUÇÃO MULTI-NÍVEL
# ==============================

niveis = {
    'produto'      : ['ID_Produto'],
    'categoria'    : ['ID_Categoria'],
    'canal'        : ['ID_Canal'],
    'produto_canal': ['ID_Produto', 'ID_Canal']
}

df_final = pd.DataFrame()

for nome, cols in niveis.items():
    df_prep = preparar_dados(df, cols)
    df_res  = rodar_backtest(df_prep, cols, nome)
    df_final = pd.concat([df_final, df_res], ignore_index=True)



🚀 Backtest nível: produto
📅 2013 | MAE: 20.75 | Erro %: 47.81
📅 2014 | MAE: 20.61 | Erro %: 47.10
📅 2015 | MAE: 20.47 | Erro %: 45.78
📅 2016 | MAE: 20.67 | Erro %: 47.50
📅 2017 | MAE: 20.53 | Erro %: 46.93
📅 2018 | MAE: 20.14 | Erro %: 45.78
📅 2019 | MAE: 20.65 | Erro %: 45.82
📅 2020 | MAE: 20.73 | Erro %: 45.89
📅 2021 | MAE: 21.09 | Erro %: 47.81

🚀 Backtest nível: categoria
📅 2013 | MAE: 136.08 | Erro %: 9.22
📅 2014 | MAE: 144.40 | Erro %: 9.67
📅 2015 | MAE: 125.36 | Erro %: 8.26
📅 2016 | MAE: 155.09 | Erro %: 10.48
📅 2017 | MAE: 124.49 | Erro %: 8.43
📅 2018 | MAE: 124.62 | Erro %: 8.36
📅 2019 | MAE: 157.26 | Erro %: 10.22
📅 2020 | MAE: 138.87 | Erro %: 9.03
📅 2021 | MAE: 160.95 | Erro %: 10.76

🚀 Backtest nível: canal
📅 2013 | MAE: 263.22 | Erro %: 5.95
📅 2014 | MAE: 225.88 | Erro %: 5.04
📅 2015 | MAE: 209.11 | Erro %: 4.59
📅 2016 | MAE: 293.88 | Erro %: 6.62
📅 2017 | MAE: 272.07 | Erro %: 6.14
📅 2018 | MAE: 271.28 | Erro %: 6.06
📅 2019 | MAE: 281.85 | Erro %: 6.11
📅 2020 | MAE: 27

In [6]:
# ==============================
# RESULTADO CONSOLIDADO
# ==============================
print("\n📊 Resultado consolidado:")
print(df_final.groupby('Nivel')[['MAE', 'Erro_%']].mean().round(2))

df_final['Data_Execucao'] = datetime.now()
df_final.to_sql("backtest_multinivel", engine, if_exists="replace", index=False)
print("\n✅ Salvo no banco: backtest_multinivel")



📊 Resultado consolidado:
                  MAE  Erro_%
Nivel                        
canal          264.69    5.88
categoria      140.79    9.38
produto         20.63   46.71
produto_canal   13.77   52.32

✅ Salvo no banco: backtest_multinivel
